In [1]:
!pip install numpy

DEPRECATION: Loading egg at c:\users\lovep\appdata\local\programs\python\python311\lib\site-packages\pysoundfile-0.9.0.post1-py3.11-win-amd64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at c:\users\lovep\appdata\local\programs\python\python311\lib\site-packages\sstv-0.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# 1. 读取彩色图片
image = Image.open('test3.png')  # 替换为你的图片路径
img_array = np.array(image)  # 形状为 (height, width, 3)

# 分离RGB通道
r_channel = img_array[:, :, 0]  # 红色分量
g_channel = img_array[:, :, 1]  # 绿色分量
b_channel = img_array[:, :, 2]  # 蓝色分量

# 将通道转换为图像
r_image = Image.fromarray(r_channel, mode='L')
g_image = Image.fromarray(g_channel, mode='L')
b_image = Image.fromarray(b_channel, mode='L')

# 使用matplotlib创建子图并排显示
fig, axes = plt.subplots(1, 4, figsize=(15, 5))

# 显示原图
axes[0].imshow(image)
axes[0].set_title('Original Image')
axes[0].axis('off')

# 显示红色通道
axes[1].imshow(r_channel, cmap='Reds')
axes[1].set_title('Red Channel')
axes[1].axis('off')

# 显示绿色通道
axes[2].imshow(g_channel, cmap='Greens')
axes[2].set_title('Green Channel')
axes[2].axis('off')

# 显示绿色通道
axes[3].imshow(b_channel, cmap='Blues')
axes[3].set_title('Blue Channel')
axes[3].axis('off')

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'test3.png'

In [28]:
def svd(channel):
    """
    对对称矩阵进行特征值分解
    """
    new_channel = channel.T @ channel

    # 对于对称矩阵，使用eigh更高效且保证特征值为实数
    eigenvalues, eigenvectors = np.linalg.eigh(new_channel)

    # 由于对称矩阵的特征值保证为实数，这里取绝对值
    eigenvalues_abs = np.abs(eigenvalues)

    # 按特征值降序排列
    idx = np.argsort(eigenvalues_abs)[::-1]
    eigenvalues_abs = eigenvalues_abs[idx]
    eigenvectors = eigenvectors[:, idx]

    singular_values = np.sqrt(eigenvalues_abs)

    # 对于对称矩阵，特征向量已经正交，但我们可以验证并确保正交性
    orthonormal_vectors = eigenvectors

    # 计算左奇异值
    Un = (channel @ orthonormal_vectors) / singular_values

    return Un, singular_values, orthonormal_vectors.T

u, s, v = svd(r_channel)

In [3]:
# 选取第 k 个特征值组成图片
def process(r_channel, g_channel, b_channel, k=None, mode=None):

    r_u, r_s, r_v = svd(r_channel)
    g_u, g_s, g_v = svd(g_channel)
    b_u, b_s, b_v = svd(b_channel)

    if k is None:
        k = list(range(r_s.shape[0] + 1))

    ks = k
    if not isinstance(k, list):
        ks = [k]

    for k in ks:

        r_s_c = r_s.copy()
        g_s_c = g_s.copy()
        b_s_c = b_s.copy()

        r_s_c[k:] = 0
        g_s_c[k:] = 0
        b_s_c[k:] = 0

        r_channel = r_u @ np.diag(r_s_c) @ r_v
        g_channel = g_u @ np.diag(g_s_c) @ g_v
        b_channel = b_u @ np.diag(b_s_c) @ b_v

        channel = np.stack((r_channel, g_channel, b_channel), axis=-1).astype(np.uint8)

        # 使用matplotlib创建子图并排显示
        fig, axes = plt.subplots(1, 4, figsize=(15, 5))

        # 显示原图
        axes[0].imshow(Image.fromarray(channel, mode))
        axes[0].set_title(f'Original Image k={k}')
        axes[0].axis('off')

        if mode == 'YCbCr':
            # 显示 Y 通道（亮度）
            axes[1].imshow(r_channel, cmap='gray')  # Y 通道用灰度图显示
            axes[1].set_title('Y Channel (Luminance)')
            axes[1].axis('off')

            # 显示 Cb 通道（蓝色色差）
            axes[2].imshow(g_channel, cmap='RdYlBu_r')  # 使用适合显示色差的 colormap
            axes[2].set_title('Cb Channel (Blue Difference)')
            axes[2].axis('off')

            # 显示 Cr 通道（红色色差）
            axes[3].imshow(b_channel, cmap='RdYlBu_r')  # 使用适合显示色差的 colormap
            axes[3].set_title('Cr Channel (Red Difference)')
            axes[3].axis('off')
        else:
            # 显示红色通道
            axes[1].imshow(r_channel, cmap='Reds')
            axes[1].set_title('Red Channel')
            axes[1].axis('off')

            # 显示绿色通道
            axes[2].imshow(g_channel, cmap='Greens')
            axes[2].set_title('Green Channel')
            axes[2].axis('off')

            # 显示绿色通道
            axes[3].imshow(b_channel, cmap='Blues')
            axes[3].set_title('Blue Channel')
            axes[3].axis('off')



        fig.savefig(f'./result/test{k}.png')

        plt.close()


In [45]:
image = Image.open('test3.png')
img_array = np.array(image)

# 分离RGB通道
r_channel = img_array[:, :, 0]  # 红色分量
g_channel = img_array[:, :, 1]  # 绿色分量
b_channel = img_array[:, :, 2]  # 蓝色分量

process(r_channel, g_channel, b_channel)

In [59]:
import cv2
import os

def images_to_video(image_folder, output_video, fps=24, k=316):
    """
    将图片文件夹中的图片拼接为视频

    Args:
        image_folder: 图片文件夹路径
        output_video: 输出视频路径
        fps: 帧率，默认24
    """
    # 获取文件夹中所有图片文件
    images = [f"test{i}.png" for i in range(0, k + 1)]

    if not images:
        print("未找到图片文件")
        return

    # 读取第一张图片获取尺寸
    first_image = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = first_image.shape

    # 创建视频写入对象
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # MP4格式
    video = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    # 逐帧写入视频
    for image_name in images:
        img_path = os.path.join(image_folder, image_name)
        img = cv2.imread(img_path)
        video.write(img)
        print(f"已处理: {image_name}")

    # 释放资源
    video.release()
    cv2.destroyAllWindows()
    print(f"视频已保存至: {output_video}")

# 使用示例
if __name__ == "__main__":
    image_folder = "result"  # 图片文件夹
    output_video = "output_video.mp4"  # 输出视频文件名

    images_to_video(image_folder, output_video, 24, 316)

已处理: test0.png
已处理: test1.png
已处理: test2.png
已处理: test3.png
已处理: test4.png
已处理: test5.png
已处理: test6.png
已处理: test7.png
已处理: test8.png
已处理: test9.png
已处理: test10.png
已处理: test11.png
已处理: test12.png
已处理: test13.png
已处理: test14.png
已处理: test15.png
已处理: test16.png
已处理: test17.png
已处理: test18.png
已处理: test19.png
已处理: test20.png
已处理: test21.png
已处理: test22.png
已处理: test23.png
已处理: test24.png
已处理: test25.png
已处理: test26.png
已处理: test27.png
已处理: test28.png
已处理: test29.png
已处理: test30.png
已处理: test31.png
已处理: test32.png
已处理: test33.png
已处理: test34.png
已处理: test35.png
已处理: test36.png
已处理: test37.png
已处理: test38.png
已处理: test39.png
已处理: test40.png
已处理: test41.png
已处理: test42.png
已处理: test43.png
已处理: test44.png
已处理: test45.png
已处理: test46.png
已处理: test47.png
已处理: test48.png
已处理: test49.png
已处理: test50.png
已处理: test51.png
已处理: test52.png
已处理: test53.png
已处理: test54.png
已处理: test55.png
已处理: test56.png
已处理: test57.png
已处理: test58.png
已处理: test59.png
已处理: test60.png
已处理: test61.png
已处理: test62.png
已处

In [58]:
image = Image.open('test3.png').convert('YCbCr')
img_array = np.array(image)

# 分离RGB通道
r_channel = img_array[:, :, 0]  # 红色分量
g_channel = img_array[:, :, 1]  # 绿色分量
b_channel = img_array[:, :, 2]  # 蓝色分量

process(r_channel, g_channel, b_channel, None, 'YCbCr')

In [19]:
a = np.random.rand(3, 5)
print(a.T.shape)
a.T @ a.reshape(3, 5)

(5, 3)


array([[0.90776306, 0.19415231, 0.76613534, 1.03700821, 0.91971999],
       [0.19415231, 0.76964658, 0.37831609, 0.80701017, 0.68003928],
       [0.76613534, 0.37831609, 0.80177726, 1.08359462, 0.89335742],
       [1.03700821, 0.80701017, 1.08359462, 1.66910798, 1.42926165],
       [0.91971999, 0.68003928, 0.89335742, 1.42926165, 1.25958628]])